# Rainfall Pattern Analysis - Choma, Zambia
## Exploratory Data Analysis

This notebook provides exploratory analysis of the Choma meteorological station data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

## 1. Load Raw Data

In [ ]:
# Load monthly data
data_dir = Path('../choma station data')

rainfall = pd.read_excel(data_dir / 'rainfall.xlsx')
humidity = pd.read_excel(data_dir / 'humidity.xlsx')
max_temp = pd.read_excel(data_dir / 'max_temp.xlsx')
min_temp = pd.read_excel(data_dir / 'mini_temp.xlsx')

print("Data loaded successfully!")
print(f"Rainfall shape: {rainfall.shape}")
print(f"Years covered: {rainfall['YY'].min()} - {rainfall['YY'].max()}")

## 2. Data Overview

In [ ]:
# Display first few rows
print("Rainfall Data:")
display(rainfall.head())

print("\nBasic Statistics:")
display(rainfall.describe())

## 3. Missing Data Analysis

In [ ]:
# Check missing values
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

datasets = [
    (rainfall, 'Rainfall', axes[0, 0]),
    (humidity, 'Humidity', axes[0, 1]),
    (max_temp, 'Max Temperature', axes[1, 0]),
    (min_temp, 'Min Temperature', axes[1, 1])
]

for df, name, ax in datasets:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        missing.plot(kind='bar', ax=ax, color='coral')
        ax.set_title(f'{name} - Missing Values')
        ax.set_ylabel('Count')
    else:
        ax.text(0.5, 0.5, 'No Missing Values', 
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{name} - Missing Values')

plt.tight_layout()
plt.show()

## 4. Rainfall Patterns

In [ ]:
# Annual rainfall totals
rainfall_annual = rainfall.iloc[:, 2:].sum(axis=1)
rainfall['Annual_Total'] = rainfall_annual

plt.figure(figsize=(14, 6))
plt.plot(rainfall['YY'], rainfall['Annual_Total'], marker='o', linewidth=2)
plt.axhline(rainfall['Annual_Total'].mean(), color='r', linestyle='--', 
            label=f'Mean: {rainfall["Annual_Total"].mean():.1f} mm')
plt.xlabel('Year')
plt.ylabel('Annual Rainfall (mm)')
plt.title('Annual Rainfall Totals - Choma Station (1990-2023)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Mean annual rainfall: {rainfall['Annual_Total'].mean():.2f} mm")
print(f"Std deviation: {rainfall['Annual_Total'].std():.2f} mm")
print(f"Min: {rainfall['Annual_Total'].min():.2f} mm ({rainfall.loc[rainfall['Annual_Total'].idxmin(), 'YY']})")
print(f"Max: {rainfall['Annual_Total'].max():.2f} mm ({rainfall.loc[rainfall['Annual_Total'].idxmax(), 'YY']})")

## 5. Seasonal Patterns

In [ ]:
# Monthly average rainfall
monthly_avg = rainfall.iloc[:, 2:14].mean()

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

plt.figure(figsize=(12, 6))
plt.bar(months, monthly_avg, color='steelblue', alpha=0.7)
plt.xlabel('Month')
plt.ylabel('Average Rainfall (mm)')
plt.title('Average Monthly Rainfall - Choma Station')
plt.grid(True, alpha=0.3, axis='y')
plt.show()

# Identify rainy season
rainy_months = monthly_avg[monthly_avg > 50].index.tolist()
print(f"\nRainy season months (>50mm): {[months[i-1] for i in rainy_months]}")

## 6. Temperature Analysis

In [ ]:
# Monthly temperature patterns
max_temp_monthly = max_temp.iloc[:, 2:14].mean()
min_temp_monthly = min_temp.iloc[:, 2:14].mean()

plt.figure(figsize=(12, 6))
plt.plot(months, max_temp_monthly, marker='o', label='Max Temp', linewidth=2, color='red')
plt.plot(months, min_temp_monthly, marker='o', label='Min Temp', linewidth=2, color='blue')
plt.fill_between(range(12), min_temp_monthly, max_temp_monthly, alpha=0.2)
plt.xlabel('Month')
plt.ylabel('Temperature (°C)')
plt.title('Average Monthly Temperature Range - Choma Station')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Correlation Analysis

In [ ]:
# Create correlation dataset
corr_data = pd.DataFrame({
    'Rainfall': rainfall.iloc[:, 2:14].mean(axis=1),
    'Max_Temp': max_temp.iloc[:, 2:14].mean(axis=1),
    'Min_Temp': min_temp.iloc[:, 2:14].mean(axis=1),
    'Humidity': humidity.iloc[:, 2:14].mean(axis=1)
})

# Correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(corr_data.corr(), annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1)
plt.title('Correlation Matrix - Annual Averages')
plt.show()

print("\nCorrelation with Rainfall:")
print(corr_data.corr()['Rainfall'].sort_values(ascending=False))

## 8. Trend Analysis

In [ ]:
# Linear trend in rainfall
from scipy import stats

years = rainfall['YY'].values
annual_rain = rainfall['Annual_Total'].values

slope, intercept, r_value, p_value, std_err = stats.linregress(years, annual_rain)

plt.figure(figsize=(12, 6))
plt.scatter(years, annual_rain, alpha=0.6, s=100)
plt.plot(years, slope * years + intercept, 'r--', linewidth=2, 
         label=f'Trend: {slope:.2f} mm/year')
plt.xlabel('Year')
plt.ylabel('Annual Rainfall (mm)')
plt.title('Rainfall Trend Analysis')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Trend: {slope:.2f} mm/year")
print(f"R-squared: {r_value**2:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    if slope > 0:
        print("Significant increasing trend detected")
    else:
        print("Significant decreasing trend detected")
else:
    print("No significant trend detected")

## 9. Rainfall Distribution

In [ ]:
# Distribution of monthly rainfall
all_monthly_rainfall = rainfall.iloc[:, 2:14].values.flatten()
all_monthly_rainfall = all_monthly_rainfall[~np.isnan(all_monthly_rainfall)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(all_monthly_rainfall, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Monthly Rainfall (mm)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Monthly Rainfall')
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(all_monthly_rainfall, vert=True)
axes[1].set_ylabel('Monthly Rainfall (mm)')
axes[1].set_title('Box Plot of Monthly Rainfall')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean monthly rainfall: {all_monthly_rainfall.mean():.2f} mm")
print(f"Median: {np.median(all_monthly_rainfall):.2f} mm")
print(f"Std deviation: {all_monthly_rainfall.std():.2f} mm")

## 10. Key Insights

Based on the analysis above, document your key findings:

1. **Rainfall Patterns:**
   - Rainy season typically occurs from...
   - Average annual rainfall is...
   - Variability is...

2. **Temperature Patterns:**
   - Hottest months are...
   - Coolest months are...
   - Temperature range is...

3. **Trends:**
   - Long-term rainfall trend shows...
   - Climate change indicators...

4. **Correlations:**
   - Rainfall is most correlated with...
   - Key predictive features are...

These insights will help justify your model design and feature selection!